# RNA3D thesis composite-TBM submission

Offline production inference using the same temporal-safe/composite-search code as the local experiments.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tarfile
import time

import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/rna3d_work')

manifests = list(INPUT_ROOT.rglob('bundle_manifest.json'))
archives = list(INPUT_ROOT.rglob('rna3d_bundle.tar.gz'))
if manifests:
    BUNDLE = manifests[0].parent
else:
    assert archives, sorted(str(p) for p in INPUT_ROOT.glob('*'))
    BUNDLE = Path('/kaggle/working/rna3d_bundle')
    if BUNDLE.exists():
        shutil.rmtree(BUNDLE)
    BUNDLE.mkdir(parents=True)
    with tarfile.open(archives[0], 'r:gz') as archive:
        archive.extractall(BUNDLE)
competition_roots = [
    p.parent for p in INPUT_ROOT.rglob('test_sequences.csv')
    if (p.parent / 'sample_submission.csv').is_file()
]
assert competition_roots, sorted(str(p) for p in INPUT_ROOT.glob('*'))
COMPETITION = competition_roots[0]
SITE_PACKAGES = Path('/kaggle/working/rna3d_site_packages')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps',
    '--find-links', str(BUNDLE / 'wheels'), '--target', str(SITE_PACKAGES),
    'biopython'
], check=True)
sys.path.insert(0, str(SITE_PACKAGES))
sys.path.insert(0, str(BUNDLE))
sys.path.insert(0, str(BUNDLE / 'src'))
RUNTIME_MMSEQS = Path('/kaggle/working/mmseqs')
shutil.copy2(BUNDLE / 'bin/mmseqs', RUNTIME_MMSEQS)
RUNTIME_MMSEQS.chmod(0o755)
os.environ['LD_LIBRARY_PATH'] = str(BUNDLE / 'lib')
print('bundle ready:', BUNDLE, 'files:', len(list(BUNDLE.rglob('*'))))

In [ ]:
from kaggle.inference_pipeline import run_inference
from rna3d.template import mmseqs_search
mmseqs_search.mmseqs_bin = lambda: str(RUNTIME_MMSEQS)

test = pd.read_csv(COMPETITION / 'test_sequences.csv')
sample = pd.read_csv(COMPETITION / 'sample_submission.csv')
started = time.time()
submission = run_inference(
    test,
    BUNDLE,
    work_dir=WORK,
    sample_submission=sample,
    steps=300,
    max_len=1000,
)
output = Path('/kaggle/working/submission.csv')
submission.to_csv(output, index=False)
assert submission.shape == sample.shape, (submission.shape, sample.shape)
assert submission['ID'].tolist() == sample['ID'].tolist()
assert not submission.isna().any().any()
print(f'wrote {output}: rows={len(submission)}, targets={len(test)}, seconds={time.time()-started:.1f}')

In [ ]:
submission.head(3)